# 10.7 模型怎麼知道圖片放在哪？


問題文字是「這張圖是什麼顏色」，圖片卻不是普通字詞。我們在序列放一個圖片佔位標記，表示這裡要插入視覺特徵。假如標記原本佔1格、視覺特徵有3格，後面的回答就要整體往後移動2格；答案標籤若沒有同步移動，會把正確文字放到錯誤位置。

前置是[10.6接頭輸出](https://birdhackor.github.io/tiny-perceptron-vlm/10.6.html)與[7.4下一位置的答案對齊](https://birdhackor.github.io/tiny-perceptron-vlm/7.4.html)。本節的序列先是完整未錯位的輸入和標籤，插入後才做一次shift，指輸入去最後一格、目標去第一格，讓每個輸入預測下一格。圖片位置標籤填`-100`，表示這些位置不直接猜文字，但仍可作為回答的上下文。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from torch import nn
from tiny_perceptron.multimodal import expand_modalities

embedding = nn.Embedding(264, 8)
ids = torch.tensor([1, 5, 4, 73, 2])
labels = torch.tensor([-100, -100, -100, 73, 2])
features = {5: torch.randn(3, 8)}
x, y = expand_modalities(ids, labels, embedding, features, {5})
print("輸入特徵", tuple(x.shape))
print("下一格目標", y.tolist())

輸入編號1表示開頭、5表示圖片、4表示助手角色、73是示範答案片段、2表示結束。數字73這裡只是已有詞表中的合法編號，不代表模型已會回答顏色。`nn.Embedding(264,8)`建立264列查表，可查編號0到263，每列有8個數字；`embedding`把普通編號轉換成8維向量，`features`給圖片標記5提供三條8維特徵；`{5}`告訴工具哪些編號需要展開。

展開後完整目標是`[-100,-100,-100,-100,-100,73,2]`：開頭、三條視覺特徵、助手角色都不算文字代價。再去掉第一個目標，與去掉最後一條輸入對應，輸出`x`形狀`(1,6,8)`，即批次中1筆樣本、6個輸入位置、每位置8個特徵，`y`為`[[-100,-100,-100,-100,73,2]]`。答案73的目標位在索引4，對應輸入中的助手角色，因此首個答案仍由它前面的上下文預測。

交給後續文字模型的是對齊後這6格輸入，因此[位置編號](https://birdhackor.github.io/tiny-perceptron-vlm/4.1.html)是0到5，指格子順序，與代表內容的詞表編號1、5、4、73、2不同。[防偷看的注意力遮罩](https://birdhackor.github.io/tiny-perceptron-vlm/3.6.html)也要配合這6格，形成6×6的表：每列是一個讀取位置，每欄是一個被讀取位置，只允許讀取自己與之前的格子。這些設定不能沿用原來5格的長度，也不能誤用完整未對齊7格的長度。

圖中第一列是五格原序列，第二列把圖片一格換成三格，形成七格。下面先列完整目標，再用箭頭表示只去掉第一個目標一次，得到六格的下一位置目標。73仍由前面的助手位置預測；淡橘色三格表示視覺上下文，不是三個要輸出的字。


In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/multimodal_expand_image.svg")))

只調整輸入而沿用舊標籤，可能不報錯卻把錯誤位置當答案，訓練意義就變了。圖片格不計代價不等於看不見，這點可回看[7.5](https://birdhackor.github.io/tiny-perceptron-vlm/7.5.html)。

注意y的索引4雖放著73，它不是要模型在輸入第4格直接看到73再抄。展開後輸入的第4格是助手角色，目標正好在其後，這才是「預測下一格」。最後一個輸入是73、目標是結束2。若先將原標籤shift、再展開圖片，就容易把角色、圖片或答案錯配；工具要求未shift的完整序列，就是為了讓這個責任只發生一次。

練習只把`randn(3,8)`改成`randn(5,8)`，先預測輸出長度8，目標中前六格忽略、末兩格仍73與2，再執行核對。回答內容沒變，移動的是它所在的位置。
